<a href="https://colab.research.google.com/github/eslam-adel-141/stroke-risk-app/blob/main/notebooks/stroke_risk_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🫀 Stroke Risk Prediction: Regression & Classification

This notebook builds two models from symptom data and age:
- **Regression:** predicts the stroke risk as a percentage (`Stroke Risk (%)`)
- **Classification:** predicts whether a person is `At Risk (Binary)`

**Workflow:** Setup → Data Loading → Cleaning → EDA → Regression → Classification → Final Models & Export


## 1. Setup
Libraries and global settings.

In [ ]:
# Imports: data handling, plotting, and the scikit-learn tools used for both the regression and classification tasks
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import math
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [ ]:
# Global setup: dark plot theme and a fixed random seed for reproducible results
plt.style.use('dark_background')
np.random.seed(42)
print("✅ Ready!")

## 2. Data Loading
Download the dataset from Kaggle and load it into a DataFrame.

In [ ]:
# Download the Stroke Risk Prediction dataset from Kaggle (via kagglehub)
import kagglehub

path = kagglehub.dataset_download("mahatiratusher/stroke-risk-prediction-dataset")

print("Path to dataset files:", path)

In [ ]:
# List the files that were downloaded
print(os.listdir(path))

In [ ]:
# Load the CSV into a DataFrame and preview the first 10 rows
csv_path = os.path.join(path, "stroke_risk_dataset.csv")

df = pd.read_csv(csv_path)

df.head(10)

## 3. Data Inspection & Cleaning
Check data types, missing values, summary statistics, and duplicates.

In [ ]:
# Inspect column types and check for missing values
df.info()

In [ ]:
# Summary statistics for every column (mean, min, max, quartiles)
df.describe()

In [ ]:
# Count fully duplicated rows
df.duplicated().sum()

In [ ]:
# Remove duplicate rows and confirm none are left
df = df.drop_duplicates()

print(df.duplicated().sum())

## 4. Exploratory Data Analysis (EDA)

### 4.1 Feature Distributions

In [ ]:
# EDA: histogram + KDE for every column (figure is saved as all_distributions.png)
cols = df.columns.tolist()
num_cols = len(cols)

n_rows = math.ceil(num_cols / 4)

fig, axes = plt.subplots(n_rows, 4, figsize=(18, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribution of {col}', fontsize=10)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('all_distributions.png', dpi=120)
plt.show()

### 4.2 Outlier Analysis
Detect outliers with the IQR rule and decide how to handle them.

In [ ]:
# Helper that detects outliers with the IQR rule, then report outlier counts per column
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    return outliers, lower, upper

for col in cols:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    print(f"\n{col}: {len(outliers)} outliers | range=[{lower:.2f}, {upper:.2f}]")

In [ ]:
# Boxplots of Age and Stroke Risk (%) to visualize outliers
target_cols = ['Age', 'Stroke Risk (%)']

fig, axes = plt.subplots(1, len(target_cols), figsize=(6 * len(target_cols), 4))

for i, col in enumerate(target_cols):
    sns.boxplot(x=df[col], ax=axes[i], color='cyan')
    axes[i].set_title(f'Boxplot of {col}', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Plot the IQR bounds over the Stroke Risk (%) distribution to see which values would count as outliers
Q1 = df['Stroke Risk (%)'].quantile(0.25)
Q3 = df['Stroke Risk (%)'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=df['Stroke Risk (%)'], ax=axes[0], color='cyan')
axes[0].set_title('Boxplot of Stroke Risk (%) - Outliers Highlighted')

sns.histplot(df['Stroke Risk (%)'], kde=True, ax=axes[1], color='steelblue')
axes[1].axvline(lower, color='yellow', linestyle='--', label=f'Lower Limit ({lower:.2f})')
axes[1].axvline(upper, color='red', linestyle='--', label=f'Upper Limit ({upper:.2f})')
axes[1].set_title('Distribution of Stroke Risk (%) with IQR Bounds')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Decision: keep the high Stroke Risk (%) rows. They are genuine high-risk cases, not noise
print(df.shape)
print((df['Stroke Risk (%)'] > upper).sum(), "rows above the IQR upper bound (kept)")

### 4.3 Label Analysis (Data Leakage Check)
`At Risk (Binary)` is derived from `Stroke Risk (%)`, so each column must be excluded when the other is the target.

In [ ]:
# Verify how the binary label is built: At Risk = 1 exactly when Stroke Risk (%) >= 50,
# so the label is derived from the regression target
print(df.groupby('At Risk (Binary)')['Stroke Risk (%)'].agg(['min', 'max', 'count']))

### 4.4 Relationships & Correlation

In [ ]:
# Scatter / pair plot of Age vs Stroke Risk (%), colored by the At Risk label
features_subset = ['Age', 'Stroke Risk (%)']
df_subset = df[features_subset].sample(1000, random_state=42)

pair_plot = sns.pairplot(
    df.sample(1000, random_state=42),
    vars=features_subset,
    hue='At Risk (Binary)',
    palette='Set2',
    diag_kind='hist',
    plot_kws={'alpha': 0.5, 's': 20}
)

pair_plot.fig.suptitle('Scatter Plot Matrix: Age vs Stroke Risk (%)', y=1.02, fontsize=12)
plt.show()

In [ ]:
# Correlation heatmap of all columns
corr_matrix = df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of each symptom with Stroke Risk (%); At Risk is excluded to avoid leakage
fig, ax = plt.subplots(figsize=(12, 6))
target_corr = corr_matrix['Stroke Risk (%)'].drop(['Stroke Risk (%)', 'At Risk (Binary)']).sort_values()
colors = ['red' if x < 0 else 'green' for x in target_corr]
target_corr.plot(kind='barh', color=colors, ax=ax, alpha=0.8)
ax.set_xlabel('Correlation Coefficient', fontsize=12)
ax.set_ylabel('Features', fontsize=12)
ax.set_title('Feature Correlation with Stroke Risk (%)', fontsize=14)
ax.axvline(0, color='white', linestyle='--', linewidth=1)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### 4.5 Multicollinearity Check (VIF)

In [ ]:
# Multicollinearity check: Variance Inflation Factor (VIF) per feature (values near 1 mean no collinearity)
def compute_vif(X):
    """Compute VIF for each feature"""
    vif_data = []
    for i in range(X.shape[1]):
        # Use all other features to predict feature i
        X_i = X[:, i]
        X_others = np.delete(X, i, axis=1)

        # Fit linear regression
        lr = LinearRegression()
        lr.fit(X_others, X_i)

        # Compute R²
        r_squared = lr.score(X_others, X_i)

        # VIF = 1 / (1 - R²)
        vif = 1 / (1 - r_squared) if r_squared < 0.999 else np.inf
        vif_data.append(vif)

    return np.array(vif_data)


X_vif = df.drop(columns=['Stroke Risk (%)', 'At Risk (Binary)'])
vif_values = compute_vif(X_vif.values)

print("Feature                          VIF")
print("-" * 42)
for name, vif in zip(X_vif.columns, vif_values):
    print(f"{name:32s}{vif:8.2f}")

## 5. Regression: Predicting Stroke Risk (%)

### 5.1 Data Preparation
Feature selection, train/test split, and scaling.

In [ ]:
# Regression setup: features = symptoms + Age. 'At Risk (Binary)' is dropped because it is
# derived from the target (data leakage). 80/20 train/test split
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Stroke Risk (%)', 'At Risk (Binary)'])
y = df['Stroke Risk (%)']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
    )
print(f"\nData split:")
print(f"  Training:  {X_train.shape[0]} samples")
print(f"  Test Set:  {X_test.shape[0]} samples")
print(f"  Features:  {X_train.shape[1]}")

In [ ]:
# Standardize the features (scaler fitted on the training set only), needed by the linear models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 5.2 Model Training & Comparison
Five regression models compared on the test set (MSE, RMSE, MAE, R²).

In [ ]:
# Train 5 regression models and compare them on the test set (MSE, RMSE, MAE, R²)
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(random_state=42, max_depth=10),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    if "Regression" in name:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({
        "Model": name,
        "MSE": round(mse, 4),
        "RMSE": round(rmse, 4),
        "MAE": round(mae, 4),
        "R2 Score": round(r2, 4)
    })

results_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
results_df

In [ ]:
# Bar chart comparing the regression models by R²plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="R2 Score", y="Model", palette="viridis")
plt.title("Model Comparison - R2 Score (Higher is Better)")
plt.xlim(0, 1)
plt.show()

### 5.3 Cross-Validation
5-fold cross-validation for a more reliable comparison.

In [ ]:
# 5-fold cross-validation on the training set for a more reliable model comparison
from sklearn.model_selection import cross_validate

cv_results = []

for name, model in models.items():
    X_data = X_train_scaled if "Regression" in name else X_train

    scores = cross_validate(
        model,
        X_data,
        y_train,
        cv=5,
        scoring=['neg_mean_squared_error', 'r2'],
        n_jobs=-1
    )

    mean_mse = -scores['test_neg_mean_squared_error'].mean()
    mean_r2 = scores['test_r2'].mean()

    cv_results.append({
        "Model": name,
        "CV Mean MSE": round(mean_mse, 4),
        "CV Mean R2": round(mean_r2, 4)
    })

cv_df = pd.DataFrame(cv_results).sort_values(by="CV Mean R2", ascending=False)
cv_df

### 5.4 Model Diagnostics
Actual vs predicted values and residuals for the selected models.

In [ ]:
# Linear Regression: actual vs predicted values and residuals distribution
best_model = LinearRegression()
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_test, y_pred, alpha=0.4, color='cyan', edgecolors='k', s=20)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Ideal Fit (y = x)')
axes[0].set_xlabel('Actual Stroke Risk (%)', fontsize=12)
axes[0].set_ylabel('Predicted Stroke Risk (%)', fontsize=12)
axes[0].set_title('Actual vs. Predicted Values', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

residuals = y_test - y_pred
sns.histplot(residuals, kde=True, ax=axes[1], color='lime')
axes[1].set_xlabel('Residuals (Actual - Predicted)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Residuals Distribution', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Linear Regression: actual vs predicted values and residuals distribution
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_test, y_pred_gb, alpha=0.3, color='cyan', edgecolors='navy', s=20, label='Data Points')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--', linewidth=1.5, label='Ideal Fit (y = x)')
axes[0].set_xlabel('Actual Stroke Risk (%)', fontsize=12)
axes[0].set_ylabel('Predicted Stroke Risk (%)', fontsize=12)
axes[0].set_title('Gradient Boosting: Actual vs. Predicted', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

residuals_gb = y_test - y_pred_gb
sns.histplot(residuals_gb, kde=True, ax=axes[1], color='lime')
axes[1].set_xlabel('Residuals (Actual - Predicted)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Gradient Boosting: Residuals Distribution', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.5 Feature Importance

In [ ]:
# Feature importance from the Linear Regression coefficients (features are scaled, so they are comparable)
feature_names = X.columns
coefficients = best_model.coef_

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=coef_df, x='Coefficient', y='Feature', palette='crest')
plt.title('Feature Importance (Linear Regression Coefficients)')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Classification: Predicting "At Risk"

### 6.1 Data Preparation
Feature selection, stratified train/test split, and scaling.

In [ ]:
# Classification setup: target = At Risk (Binary). Stroke Risk (%) is removed (leakage).
# Stratified 80/20 split + feature scaling
X = df.drop(columns=['At Risk (Binary)', 'Stroke Risk (%)'])
y = df['At Risk (Binary)']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nData split:")
print(f"  Training:  {X_train.shape[0]} samples")
print(f"  Test Set:  {X_test.shape[0]} samples")

### 6.2 Model Training & Comparison
Four classifiers compared using Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

In [ ]:
# Train 4 classifiers and compare them (Accuracy, Precision, Recall, F1, ROC-AUC)

clf_models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=10),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

clf_results = []

for name, model in clf_models.items():
    X_tr = X_train_scaled if name == "Logistic Regression" else X_train
    X_te = X_test_scaled if name == "Logistic Regression" else X_test

    model.fit(X_tr, y_train)
    preds = model.predict(X_te)
    probs = model.predict_proba(X_te)[:, 1]

    clf_results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "Precision": round(precision_score(y_test, preds), 4),
        "Recall": round(recall_score(y_test, preds), 4),
        "F1-Score": round(f1_score(y_test, preds), 4),
        "ROC-AUC": round(roc_auc_score(y_test, probs), 4)
    })

clf_df = pd.DataFrame(clf_results).sort_values(by="F1-Score", ascending=False)
clf_df

### 6.3 Model Evaluation
Confusion matrices and ROC curves.

#### Logistic Regression

In [ ]:
# Logistic Regression: confusion matrix and ROC curve
from sklearn.metrics import confusion_matrix, roc_curve, auc

y_pred_log = clf_models["Logistic Regression"].predict(X_test_scaled)
y_prob_log = clf_models["Logistic Regression"].predict_proba(X_test_scaled)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Confusion Matrix
cm_log = confusion_matrix(y_test, y_pred_log)
sns.heatmap(cm_log, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not At Risk (0)', 'At Risk (1)'],
            yticklabels=['Not At Risk (0)', 'At Risk (1)'])
axes[0].set_title('Logistic Regression - Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Plot 2: ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_log)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='cyan', lw=2, label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0, 1], [0, 1], color='red', linestyle='--')
axes[1].set_title('Logistic Regression - ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Gradient Boosting

In [ ]:
# Gradient Boosting Classifier: confusion matrix and ROC curve
# 1. Predict and compute probabilities
y_pred_gb = clf_models["Gradient Boosting"].predict(X_test)
y_prob_gb = clf_models["Gradient Boosting"].predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Confusion Matrix
cm_gb = confusion_matrix(y_test, y_pred_gb)
sns.heatmap(cm_gb, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=['Not At Risk (0)', 'At Risk (1)'],
            yticklabels=['Not At Risk (0)', 'At Risk (1)'])
axes[0].set_title('Gradient Boosting - Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Plot 2: ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_gb)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='lime', lw=2, label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0, 1], [0, 1], color='red', linestyle='--')
axes[1].set_title('Gradient Boosting - ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Random Forest

In [ ]:
# Random Forest Classifier: confusion matrix and ROC curve
y_pred_rf = clf_models["Random Forest"].predict(X_test)
y_prob_rf = clf_models["Random Forest"].predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Not At Risk (0)', 'At Risk (1)'],
            yticklabels=['Not At Risk (0)', 'At Risk (1)'])
axes[0].set_title('Random Forest - Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Plot 2: ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='orange', lw=2, label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0, 1], [0, 1], color='red', linestyle='--')
axes[1].set_title('Random Forest - ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Final Models & Export
Train the final Gradient Boosting models on the symptom features only, evaluate them on a held-out test set, and export them (with `metadata.json`) for the Streamlit app.

In [ ]:
# Production step: train the final Gradient Boosting models (regression + classification) on the symptom
# features only, evaluate them on a held-out test set, save them with joblib, and export metadata.json
# (feature order, age range, metrics) used by the app. Also downloads everything as models.zip
import os, json, shutil, joblib, sklearn
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import (r2_score, mean_absolute_error,
                             accuracy_score, f1_score, roc_auc_score)

REG_TARGET, CLF_TARGET = "Stroke Risk (%)", "At Risk (Binary)"
FEATURES = [c for c in df.columns if c not in (REG_TARGET, CLF_TARGET)]
X = df[FEATURES]

# ---------- Regression ----------
Xtr, Xte, ytr, yte = train_test_split(X, df[REG_TARGET], test_size=0.2, random_state=42)
reg_model = Pipeline([("model", GradientBoostingRegressor(n_estimators=100, random_state=42))])
reg_model.fit(Xtr, ytr)
p = reg_model.predict(Xte)
reg_metrics = {"r2": r2_score(yte, p), "mae": mean_absolute_error(yte, p)}

# ---------- Classification ----------
Xtr, Xte, ytr, yte = train_test_split(X, df[CLF_TARGET], test_size=0.2,
                                      random_state=42, stratify=df[CLF_TARGET])
clf_model = Pipeline([("model", GradientBoostingClassifier(n_estimators=100, random_state=42))])
clf_model.fit(Xtr, ytr)
pred = clf_model.predict(Xte)
prob = clf_model.predict_proba(Xte)[:, 1]
clf_metrics = {"accuracy": accuracy_score(yte, pred), "f1": f1_score(yte, pred),
               "roc_auc": roc_auc_score(yte, prob)}

# ---------- Save ----------
os.makedirs("models", exist_ok=True)
joblib.dump(reg_model, "models/reg_model.joblib")
joblib.dump(clf_model, "models/clf_model.joblib")
with open("models/metadata.json", "w") as f:
    json.dump({"features": FEATURES,
               "age_min": int(df["Age"].min()),
               "age_max": int(df["Age"].max()),
               "reg_metrics": reg_metrics,
               "clf_metrics": clf_metrics}, f, indent=2)

print("Regression:", reg_metrics)
print("Classification:", clf_metrics)
print("scikit-learn version:", sklearn.__version__)

shutil.make_archive("models", "zip", "models")
from google.colab import files
files.download("models.zip")

## 8. Summary
- Final models: Gradient Boosting for both regression and classification
- Key finding: `At Risk (Binary)` equals `Stroke Risk (%) >= 50`, so the labels follow a fixed rule and the dataset appears synthetic
- Both models were exported to `models/` and are served by the Streamlit app in this repo